# 🦆 Microduck 雙足機器人強化學習 (Google Colab 雲端訓練完整手冊)

本筆記本支援在 **Google Colab (免本地算力)** 雲端環境中執行 Pollen Robotics 14-DOF Microduck 雙足機器人的 PPO 強化學習步態訓練。

---

### 🌟 本版本已修復之核心問題
1. **GPU (EGL) / CPU (OSMesa) 無頭渲染自動相容**：自動辨識 Colab 是否開啟 GPU 加速；開啟 GPU 時自動配置 NVIDIA EGL 驅動，解決 `Failed to initialize GLFW` 與無顯示器崩潰問題。
2. **補齊所有相依套件**：預先安裝 `onnx`、`imageio-ffmpeg`、`rich`，徹底解決匯出模型時 `ModuleNotFoundError: No module named 'onnx'` 的問題。
3. **In-Process 影片生成與播放**：直接在 Python 核心內執行步態錄製，生成的 MP4 採用 **H.264 (yuv420p)** 網頁標準編碼，並提供 **動態 GIF 備用方案**，保證可在瀏覽器中順暢播放。
4. **ZIP 自動檢測與互動上傳**：若未找到 `microduck_colab.zip`，會自動彈出上傳選單，避免路徑錯誤中斷。

## 🛠️ 第一步：安裝依賴套件與自動配置無頭 (Headless) 渲染驅動

In [ ]:
# 1. 安裝系統級多媒體與渲染庫
!apt-get update -qq && apt-get install -y -qq ffmpeg libosmesa6-dev libgl1-mesa-glx libgl1-mesa-dri > /dev/null

# 2. 安裝 Python 核心強化學習庫與 ONNX 支援
!pip install -q gymnasium stable-baselines3 mujoco opencv-python imageio imageio-ffmpeg onnx rich tqdm

# 3. 自動檢測 GPU 並配置最合適的 MuJoCo 無頭渲染後端 (GPU: EGL / CPU: OSMesa)
import os, sys, subprocess

has_gpu = False
try:
    has_gpu = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
except Exception:
    has_gpu = False

if has_gpu:
    icd_dir = '/usr/share/glvnd/egl_vendor.d'
    os.makedirs(icd_dir, exist_ok=True)
    icd_path = os.path.join(icd_dir, '10_nvidia.json')
    if not os.path.exists(icd_path):
        with open(icd_path, 'w') as f:
            f.write('{\n    "file_format_version" : "1.0.0",\n    "ICD" : {\n        "library_path" : "libEGL_nvidia.so.0"\n    }\n}\n')
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"
    print("⚡ 成功偵測到 NVIDIA GPU！已配置硬體加速 EGL 無頭渲染。")
else:
    os.environ["MUJOCO_GL"] = "osmesa"
    print("💻 處於 CPU 模式，已配置 OSMesa 純軟體無頭渲染。")

# 確保當前路徑包含在 Python 搜尋路徑中
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

## 📂 第二步：解壓縮專案檔案 (`microduck_colab.zip`)

> **說明**：若您尚未上傳 `microduck_colab.zip`，此儲存格會**自動彈出上傳選單**供您選擇檔案！

In [ ]:
import os, zipfile

zip_name = "microduck_colab.zip"
drive_zip = "/content/drive/MyDrive/microduck_colab.zip"

# 1. 優先從 Google Drive 讀取 (若有掛載)
if not os.path.exists(zip_name) and os.path.exists(drive_zip):
    !cp /content/drive/MyDrive/microduck_colab.zip ./microduck_colab.zip

# 2. 若本地不存在，自動自 GitHub 下載
if not os.path.exists(zip_name):
    print(f"⬇️ 正在自 GitHub 自動下載專案包 {zip_name}...")
    !wget -q --show-progress -O microduck_colab.zip https://github.com/Child-pi/microduck-private/raw/rl-locomotion/microduck_colab.zip || true

# 3. 若自動下載失敗，開啟上傳選單供手動上傳
if not os.path.exists(zip_name) or os.path.getsize(zip_name) < 1000:
    if os.path.exists(zip_name):
        os.remove(zip_name)
    print(f"⚠️ 未能自 GitHub 下載，正在開啟上傳選單供手動上傳 {zip_name}...")
    from google.colab import files
    uploaded = files.upload()

# 4. 解壓縮專案檔案
if os.path.exists(zip_name) and os.path.getsize(zip_name) > 1000:
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall("./")
    print(f"✅ 專案檔案 {zip_name} 解壓縮成功！")
else:
    raise FileNotFoundError(f"❌ 請確保 {zip_name} 已成功下載或上傳！")

# 檢查關鍵目錄結構
assert os.path.exists("envs"), "❌ 缺少 envs 目錄"
assert os.path.exists("models"), "❌ 缺少 models 目錄"
print("📁 專案檔案結構驗證全部通過！")
!ls -lh envs models


## 🧪 第三步：測試環境加載與 MuJoCo 渲染功能

In [ ]:
import gymnasium as gym
import cv2
import numpy as np
from IPython.display import Image, display

import envs  # 註冊 Microduck-v0

try:
    env = gym.make("Microduck-v0", render_mode="rgb_array", render_width=640, render_height=480)
    obs, info = env.reset(seed=42)
    frame = env.render()
    env.close()

    assert frame is not None and frame.shape == (480, 640, 3), f"異常渲染畫面尺寸: {frame.shape if frame is not None else None}"
    cv2.imwrite("test_render.png", cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    print(f"✅ MuJoCo 環境加載與無頭渲染測試成功！觀察維度: {obs.shape}，動作維度: {env.action_space.shape}")
    display(Image("test_render.png"))
except Exception as e:
    print(f"❌ 渲染測試遇到問題: {e}")
    raise e

## 🦆 第四步：啟動 PPO 強化學習雲端向量化訓練

預設並行開展 8 個獨立小鴨模擬環境，大幅加速步態探索。

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
import envs

# 1. 建立 8 個並行訓練環境 (使用在 Jupyter/Colab 中最穩定的 DummyVecEnv)
num_envs = 8
train_env = make_vec_env("Microduck-v0", n_envs=num_envs, seed=42, vec_env_cls=DummyVecEnv)

# 2. 優先載入預訓練權重 (若存在)，接續微調提升耐力
resume_checkpoint = "checkpoints/best/best_model.zip"
if os.path.exists(resume_checkpoint):
    print(f"📂 載入已訓練權重進行接續微調: {resume_checkpoint}")
    model = PPO.load(
        resume_checkpoint,
        env=train_env,
        device="auto",
        tensorboard_log="logs"
    )
    model.learning_rate = 1e-4
    model.gamma = 0.9985
else:
    print("🌱 從頭開始初始化全新 PPO 網絡")
    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        gamma=0.9985,
        batch_size=128,
        n_steps=512,
        device="auto",
        verbose=1,
        tensorboard_log="logs"
    )

# 3. 執行訓練 (例如 200,000 步，約 2~3 分鐘)
total_training_timesteps = 200_000
print(f"🚀 開始雲端強化學習訓練 (總步數: {total_training_timesteps:,})...")
model.learn(total_timesteps=total_training_timesteps, progress_bar=True)

# 4. 保存模型權重
os.makedirs("checkpoints", exist_ok=True)
output_model_path = "checkpoints/microduck_colab_trained.zip"
model.save(output_model_path)
print(f"\n✅ 訓練完成！新權重已保存至: {output_model_path}")
train_env.close()

## 🎥 第五步：評估最新模型並生成帶 HUD 遙測之步態影片

直接呼叫評估核心程式，避免子程序 GLFW 顯示器相依問題，並產生相容 HTML5 的 H.264 MP4 影片與動態 GIF。

In [ ]:
from analyze_and_record import evaluate_and_record

model_eval_path = "checkpoints/microduck_colab_trained.zip"
if not os.path.exists(model_eval_path):
    model_eval_path = "checkpoints/best/best_model.zip"

print(f"🎬 正在評估模型並錄製步態: {model_eval_path}")
duration, total_steps, telemetry = evaluate_and_record(
    model_path=model_eval_path,
    output_video="colab_walk.mp4",
    output_gif="colab_walk.gif",
    output_snapshot="colab_walk_snapshot.png",
    target_vx=0.08,
    max_steps=600,
    seed=89
)
print(f"✅ 影片錄製完成！小鴨持續行走了 {duration:.2f} 秒 ({total_steps} 步)！")

## 📺 第六步：在 Colab 網頁中直接播放小鴨步態成果！

In [ ]:
from IPython.display import HTML, Image, display
from base64 import b64encode

# 優先嘗試內嵌 MP4 播放器
if os.path.exists("colab_walk.mp4"):
    with open("colab_walk.mp4", "rb") as f:
        mp4_data = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()
    display(HTML(f'''
    <div style="text-align:center; background:#0b111e; padding:18px; border-radius:14px; max-width:700px; margin:auto;">
      <h3 style="color:#2ec4b6; margin-bottom:10px;">🦆 Microduck 雲端訓練成果影片 (640x480 HUD Telemetry)</h3>
      <video width="640" height="480" controls autoplay loop style="border-radius:10px; border:2px solid #2ec4b6;">
        <source src="{data_url}" type="video/mp4">
        您的瀏覽器不支援 HTML5 影片，請觀看下方 GIF 預覽。
      </video>
    </div>
    '''))

# 同時提供動態 GIF 備用展示 (100% 保證可顯示)
if os.path.exists("colab_walk.gif"):
    print("\n🖼️ 動態 GIF 預覽:")
    display(Image("colab_walk.gif"))

## 📦 第七步：匯出實機 ONNX 模型並一鍵下載回電腦

In [ ]:
# 匯出符合 Pollen Robotics 實機規範的 ONNX 模型
!python export_onnx.py \
  --model-path checkpoints/microduck_colab_trained.zip \
  --output-path exported/microduck_colab_policy.onnx

# 下載權重與 ONNX 檔案
try:
    from google.colab import files
    if os.path.exists("checkpoints/microduck_colab_trained.zip"):
        print("📥 下載訓練權重 (.zip):")
        files.download("checkpoints/microduck_colab_trained.zip")
    if os.path.exists("exported/microduck_colab_policy.onnx"):
        print("📥 下載 ONNX 實機模型 (.onnx):")
        files.download("exported/microduck_colab_policy.onnx")
except Exception as e:
    print(f"下載選單未觸發，您也可以直接在左側檔案面板中手動點擊下載: {e}")